# Kandinsky 6 SR — batch example

Super-resolves **several clips in one call**: the clips are encoded together and tiled the same way, and each tile step (latent upscaler, DiT, KVAE decode) runs on that tile of all clips at once, so the GPU does `B` clips in fewer, larger steps than `B` separate runs.

Requirements for a batch:
- all clips have the **same frame count and resolution** after the 24 fps resample / 5 s cap (the cell below checks it);
- memory grows with the batch — each tile step holds `tiles_batch_size × B` tiles. One 768×512 clip at ×2 peaks around 51 GiB, so start with `B = 2` on an 80 GB GPU and ×2.

For a single video see `sr_inference_example.ipynb`.

### 1. Config

`INPUT_VIDEOS` lists the clips of the batch; each gets its own output file and keeps its own audio track.

In [ ]:
from pathlib import Path

import torch
from IPython.display import Video, display

from kandinsky_sr.pipeline.config import SRConfig

DEVICE = "cuda:0"
INPUT_VIDEOS = [Path("inputs/lq_0.mp4"), Path("inputs/lq_1.mp4")]  # same resolution and length
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SR_CONFIG = SRConfig(
    # The Diffusers bundle: the π-Flow distilled DiT (2 evaluations per tile), the KVAE and the latent upscalers.
    checkpoint_path="kandinskylab/Kandinsky-6.0-VSR-distilled2steps-5s-Diffusers",
    resolution_scale=2,  # 2, 4, or 2.25 (x1.125 pixel pre-upscale + x2 tiling)
    tiles_batch_size=1,  # tiles per clip in one model call; the effective batch is tiles_batch_size × len(INPUT_VIDEOS)
    vae_backend="torch",  # "magi" needs the `magi` extra
    device=DEVICE,
)

### 2. Load the clips

Each file is read as `[T, 3, H, W]` uint8, resampled to the model's 24 fps and cut to the 5 s budget on the `1 + 8k` frame grid (what `kandy-sr` does for a single file). The batch is the stacked `[B, T, 3, H, W]` tensor.

In [ ]:
from kandinsky_sr.pipeline.video_io import clip_to_aligned_frames, read_video_tchw_uint8, resample_to_target_fps

clips, fps_values = [], []
for path in INPUT_VIDEOS:
    frames, source_fps = read_video_tchw_uint8(path)
    frames, out_fps = resample_to_target_fps(frames, source_fps)
    clips.append(clip_to_aligned_frames(frames))
    fps_values.append(out_fps)

shapes = {tuple(clip.shape) for clip in clips}
if len(shapes) != 1 or len(set(fps_values)) != 1:
    raise ValueError(f"a batch needs clips of one shape and fps, got shapes={shapes} fps={set(fps_values)}")

lq_batch = torch.stack(clips)  # [B, T, 3, H, W]
OUT_FPS = fps_values[0]
print(f"input batch: {tuple(lq_batch.shape)} {lq_batch.dtype} @ {OUT_FPS} FPS")
for path in INPUT_VIDEOS:
    display(Video(str(path), embed=False, width=480))

### 3. Load the SR pipeline

Same as the single-video example: build the models, then compile the attention kernels and the KVAE decode once.

In [ ]:
from kandinsky_sr.pipeline.components import scale_factor_for
from kandinsky_sr.pipeline.factory import load_sr_pipeline
from kandinsky_sr.pipeline.warmup import clip_base_resolution, compile_vae_decode, warmup

sr_pipe = load_sr_pipeline(SR_CONFIG, DEVICE, force=True)
print(f"loaded SR pipeline on {DEVICE}; scale={SR_CONFIG.resolution_scale}")

warmup(sr_pipe.dit, scale_factor_for(sr_pipe), DEVICE)
compile_vae_decode(sr_pipe, DEVICE, bases=[clip_base_resolution(sr_pipe, tuple(lq_batch.shape[-2:]))])

### 4. Batched inference

`video` takes the `[B, T, 3, H, W]` tensor (or a list of `[T, 3, H, W]` clips); `save_path` and `source_video` are lists with one entry per clip. The result holds `frames` as `[B, 3, T, H·scale, W·scale]` and `path` as the list of written files.

In [ ]:
scale_tag = str(SR_CONFIG.resolution_scale).replace(".", "p")
sr_paths = [OUT_DIR / f"{path.stem}_sr_{scale_tag}x_batch{index}.mp4" for index, path in enumerate(INPUT_VIDEOS)]

torch.cuda.reset_peak_memory_stats(DEVICE)
with torch.inference_mode():
    sr_result = sr_pipe(
        video=lq_batch,
        fps=OUT_FPS,
        show_progress=True,
        source_video=INPUT_VIDEOS,  # each output keeps the audio track of its own source
        save_path=sr_paths,
    )
print(f"SR output batch: {tuple(sr_result.frames.shape)}")
print(f"saved: {sr_result.path}")
print(f"peak GPU memory: {torch.cuda.max_memory_allocated(DEVICE) / 2**30:.1f} GiB")

In [ ]:
for sr_path in sr_paths:
    display(Video(str(sr_path), embed=False, width=640))